# Phase 2 -- Meta-analysis and Network Analysis with MUUMI

**Project:** MAGIC SOLUTION -- Computational pipeline for the identification of molecular targets in ischemia-reperfusion acute kidney injury (IR-AKI)  
**Input:** Phase 1 outputs (per-dataset DE gene-level results)  
**Main tool:** [MUUMI](https://github.com/fhaive/muumi) (Inkala et al., BMC Bioinformatics, 2026)  

**Datasets:**  
- **Cat. A** (IRI vs Control): GSE43974 (microarray, n=206), GSE142077 (RNA-seq, n=10), GSE126805 (RNA-seq, n=83)  
- **Cat. B** (Validation, DGF vs IGF): GSE54888 (microarray, n=54)  

---

### Notebook structure

1. **Setup and data loading**  
2. **Statistical meta-analysis** -- Ensemble meta-analysis (MUUMI Module 1)  
3. **GSEA and seed gene selection** -- Feature selection with |log2FC| > 0.379  
4. **Cat. B validation** -- Independent confirmation on GSE54888  
5. **Co-expression networks** -- Network inference (MUUMI Module 2)  
6. **Cross-platform late integration** -- NMF module aggregation (MUUMI Module 3)  
7. **ESEA** -- Edge Set Enrichment Analysis (MUUMI Module 4)  
8. **Summary**

## 1. Setup and data loading

In [ ]:
# ============================================================
# 1.1  Libraries
# ============================================================
suppressPackageStartupMessages({
  library(muumi)
  library(dplyr)
  library(tidyr)
  library(ggplot2)
  library(clusterProfiler)
  library(ReactomePA)
  library(org.Hs.eg.db)
  library(fgsea)
  library(igraph)
})

set.seed(42)
theme_set(theme_bw(base_size = 12))

In [ ]:
# ============================================================
# 1.2  Paths -- ADJUST to your folder structure
# ============================================================
BASE_DIR   <- ".."
FASE1_DIR  <- file.path(BASE_DIR, "output", "fase1")
OUT_DIR    <- file.path(BASE_DIR, "output", "fase2")
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

# Cat. A datasets (IRI vs Control)
cat_a_files <- c(
  GSE43974  = file.path(FASE1_DIR, "GSE43974",  "de_GSE43974_IRI_vs_Control.csv"),
  GSE142077 = file.path(FASE1_DIR, "GSE142077", "de_GSE142077_IRI_vs_Control.csv"),
  GSE126805 = file.path(FASE1_DIR, "GSE126805", "de_GSE126805_IRI_vs_Control.csv")
)

# Cat. B dataset (DGF vs IGF -- validation)
cat_b_files <- c(
  GSE54888 = file.path(FASE1_DIR, "GSE54888", "de_GSE54888_DGF_vs_IGF.csv")
)

# Expression matrices (for network analysis)
expr_files <- c(
  GSE43974  = file.path(FASE1_DIR, "GSE43974",  "expr_GSE43974.csv"),
  GSE142077 = file.path(FASE1_DIR, "GSE142077", "expr_GSE142077.csv.gz"),
  GSE126805 = file.path(FASE1_DIR, "GSE126805", "expr_GSE126805.csv.gz")
)

# Sample labels
label_files <- c(
  GSE43974  = file.path(FASE1_DIR, "GSE43974",  "sample_labels_GSE43974.csv"),
  GSE142077 = file.path(FASE1_DIR, "GSE142077", "sample_labels_GSE142077.csv"),
  GSE126805 = file.path(FASE1_DIR, "GSE126805", "sample_labels_GSE126805.csv")
)

In [ ]:
# ============================================================
# 1.3  Load Phase 1 DE results
# ============================================================
load_de <- function(path, dataset_id) {
  df <- read.csv(path, stringsAsFactors = FALSE)
  df$dataset <- dataset_id
  df
}

de_cat_a <- lapply(names(cat_a_files), function(id) load_de(cat_a_files[id], id))
names(de_cat_a) <- names(cat_a_files)

de_cat_b <- lapply(names(cat_b_files), function(id) load_de(cat_b_files[id], id))
names(de_cat_b) <- names(cat_b_files)

cat("== Cat. A dataset summary ==\n")
for (nm in names(de_cat_a)) {
  d <- de_cat_a[[nm]]
  n_sig <- sum(d$padj < 0.05 & abs(d$log2FoldChange) > 0.379, na.rm = TRUE)
  cat(sprintf("  %s: %d total genes, %d DEG (|log2FC|>0.379, FDR<0.05)\n", nm, nrow(d), n_sig))
}
cat("\n== Cat. B dataset summary ==\n")
for (nm in names(de_cat_b)) {
  d <- de_cat_b[[nm]]
  n_sig <- sum(d$padj < 0.05 & abs(d$log2FoldChange) > 0.379, na.rm = TRUE)
  cat(sprintf("  %s: %d total genes, %d DEG\n", nm, nrow(d), n_sig))
}

## 2. Statistical meta-analysis (MUUMI Module 1)

In [ ]:
# ============================================================
# 2.1  Build adjusted p-value matrix across datasets
# ============================================================
padj_list <- lapply(names(de_cat_a), function(id) {
  d <- de_cat_a[[id]] %>%
    dplyr::select(gene, padj) %>%
    dplyr::filter(!is.na(padj), padj > 0, padj <= 1) %>%
    dplyr::distinct(gene, .keep_all = TRUE)
  colnames(d)[2] <- paste0(id, "_adj_pval")
  d
})

# Inner join: keep genes measured in all 3 datasets
meta_df <- padj_list[[1]]
for (i in 2:length(padj_list)) {
  meta_df <- merge(meta_df, padj_list[[i]], by = "gene", all = FALSE)
}
rownames(meta_df) <- meta_df$gene
meta_df$gene <- NULL

cat(sprintf("P-value matrix: %d genes x %d datasets\n", nrow(meta_df), ncol(meta_df)))

In [ ]:
# ============================================================
# 2.2  Ensemble meta-analysis with MUUMI
#      Combines effect size and Fisher p-value methods
# ============================================================
class_vec  <- rep(1, ncol(meta_df))
origin_vec <- seq_len(ncol(meta_df))

ranked_gene_list <- run_ensembl_metanalysis(
  meta_dataframe = meta_df,
  method         = c("effect_size", "pvalue"),
  class          = class_vec,
  origin         = origin_vec,
  metric         = "median"
)

cat(sprintf("Meta-analysis complete: %d genes in ranking\n", nrow(ranked_gene_list)))
head(ranked_gene_list, 20)

In [ ]:
# ============================================================
# 2.3  Compute weighted log2FC and direction consistency
#      Weights are -log10(padj) from each dataset
# ============================================================
gene_ranking <- ranked_gene_list[, 1]

lfc_list <- lapply(names(de_cat_a), function(id) {
  d <- de_cat_a[[id]] %>%
    dplyr::select(gene, log2FoldChange, padj) %>%
    dplyr::filter(!is.na(padj)) %>%
    dplyr::distinct(gene, .keep_all = TRUE)
  colnames(d) <- c("gene", paste0("lfc_", id), paste0("padj_", id))
  d
})

lfc_merged <- lfc_list[[1]]
for (i in 2:length(lfc_list)) {
  lfc_merged <- merge(lfc_merged, lfc_list[[i]], by = "gene", all = FALSE)
}

lfc_cols  <- grep("^lfc_", colnames(lfc_merged), value = TRUE)
padj_cols <- grep("^padj_", colnames(lfc_merged), value = TRUE)
lfc_mat   <- as.matrix(lfc_merged[, lfc_cols])
padj_mat  <- as.matrix(lfc_merged[, padj_cols])
weights   <- -log10(pmax(padj_mat, 1e-300))

lfc_merged$weighted_lfc <- rowSums(lfc_mat * weights) / rowSums(weights)
lfc_merged$direction_consistency <- apply(lfc_mat, 1, function(x) {
  signs <- sign(x[x != 0])
  if (length(signs) == 0) return(0)
  max(table(signs)) / length(signs)
})

# Merge ranking with fold-change data
meta_rank <- data.frame(
  gene = gene_ranking,
  rank = seq_along(gene_ranking),
  stringsAsFactors = FALSE
)
meta_rank <- merge(meta_rank, lfc_merged[, c("gene", "weighted_lfc", "direction_consistency")],
                   by = "gene", all.x = TRUE)
meta_rank <- meta_rank[order(meta_rank$rank), ]

write.csv(meta_rank, file = file.path(OUT_DIR, "meta_analysis_ranked_gene_list.csv"),
          row.names = FALSE, quote = FALSE)
cat("Top 20 genes in meta-analytic ranking:\n")
head(meta_rank, 20)

## 3. GSEA and seed gene selection

In [ ]:
# ============================================================
# 3.1  Load Reactome pathways
# ============================================================
gmt_file <- file.path(BASE_DIR, "data", "c2.cp.reactome.v2023.2.Hs.symbols.gmt")

if (!file.exists(gmt_file)) {
  message("GMT file not found. Using msigdbr as alternative...")
  if (!requireNamespace("msigdbr", quietly = TRUE)) install.packages("msigdbr")
  library(msigdbr)
  reactome <- msigdbr(species = "Homo sapiens", category = "C2", subcategory = "CP:REACTOME")
  pathways_reactome <- split(reactome$gene_symbol, reactome$gs_name)
  use_gmt <- FALSE
} else {
  use_gmt <- TRUE
}

In [ ]:
# ============================================================
# 3.2  Pre-ranked GSEA using signed weighted log2FC
# ============================================================
if (use_gmt) {
  gsea_res <- compute_gsea(gene_list = ranked_gene_list, gmt_file = gmt_file,
                           no_permutations = 100000)
} else {
  gsea_input <- meta_rank %>%
    dplyr::filter(!is.na(weighted_lfc)) %>%
    dplyr::arrange(rank)
  ranking_vec <- gsea_input$weighted_lfc
  names(ranking_vec) <- gsea_input$gene
  ranking_vec <- sort(ranking_vec, decreasing = TRUE)
  
  gsea_res <- fgsea::fgsea(pathways = pathways_reactome, stats = ranking_vec,
                           minSize = 10, nPermSimple = 100000)
}

gsea_top <- gsea_res[order(gsea_res$pval), ]
cat("Top 15 enriched Reactome pathways:\n")
print(head(gsea_top[, c("pathway", "pval", "padj", "ES", "NES", "size")], 15))

In [ ]:
# ============================================================
# 3.3  GSEA-based threshold for seed gene selection
#      Median leading-edge position across significant pathways
# ============================================================
sig_pathways <- gsea_res[gsea_res$pval < 0.05, ]

if (nrow(sig_pathways) > 0) {
  gsea_input <- meta_rank %>%
    dplyr::filter(!is.na(weighted_lfc)) %>%
    dplyr::arrange(rank)
  ranking_vec <- gsea_input$weighted_lfc
  names(ranking_vec) <- gsea_input$gene
  ranking_vec <- sort(ranking_vec, decreasing = TRUE)
  gene_order <- names(ranking_vec)
  
  le_positions <- sapply(seq_len(nrow(sig_pathways)), function(i) {
    le_genes <- unlist(sig_pathways$leadingEdge[i])
    positions <- match(le_genes, gene_order)
    max(positions, na.rm = TRUE)
  })
  threshold <- round(median(le_positions, na.rm = TRUE))
} else {
  threshold <- round(0.3 * nrow(meta_rank))
  message("No significant pathways -- fallback to top 30%.")
}

cat(sprintf("GSEA-based threshold: top %d genes out of %d (%.1f%%)\n",
            threshold, nrow(meta_rank), 100 * threshold / nrow(meta_rank)))

In [ ]:
# ============================================================
# 3.4  Seed gene selection with stringent filters
#      |log2FC| > 0.379 (FC >= 1.3) AND direction consistency > 0.5
# ============================================================
seed_candidates <- meta_rank[meta_rank$rank <= threshold, ]

# Seed genes: consistency > 0.5 AND |log2FC| > 0.379
seed_genes <- seed_candidates %>%
  dplyr::filter(
    direction_consistency > 0.5,
    abs(weighted_lfc) > 0.379
  ) %>%
  dplyr::arrange(rank)

# High-confidence subset: consistency = 1.0 (concordant in all 3 datasets)
seed_genes_hc <- seed_genes %>%
  dplyr::filter(direction_consistency == 1.0)

cat(sprintf("Candidates from GSEA threshold: %d\n", nrow(seed_candidates)))
cat(sprintf("Seed genes (consistency>0.5, |lfc|>0.379): %d\n", nrow(seed_genes)))
cat(sprintf("  UP: %d | DOWN: %d\n",
            sum(seed_genes$weighted_lfc > 0), sum(seed_genes$weighted_lfc < 0)))
cat(sprintf("Seed genes HIGH-CONFIDENCE (consistency=1.0): %d\n", nrow(seed_genes_hc)))
cat(sprintf("  UP: %d | DOWN: %d\n",
            sum(seed_genes_hc$weighted_lfc > 0), sum(seed_genes_hc$weighted_lfc < 0)))

write.csv(seed_genes, file = file.path(OUT_DIR, "seed_genes.csv"),
          row.names = FALSE, quote = FALSE)
write.csv(seed_genes_hc, file = file.path(OUT_DIR, "seed_genes_high_confidence.csv"),
          row.names = FALSE, quote = FALSE)

cat("\nTop 20 seed genes:\n")
head(seed_genes, 20)

In [ ]:
# ============================================================
# 3.5  ORA on seed genes (up and down separately)
#      Background = all genes from the meta-analysis
# ============================================================
seed_up   <- seed_genes$gene[seed_genes$weighted_lfc > 0]
seed_down <- seed_genes$gene[seed_genes$weighted_lfc < 0]
bg_genes  <- meta_rank$gene

convert_to_entrez <- function(symbols) {
  conv <- clusterProfiler::bitr(symbols, fromType = "SYMBOL", toType = "ENTREZID",
                                OrgDb = org.Hs.eg.db)
  conv$ENTREZID
}

entrez_up   <- convert_to_entrez(seed_up)
entrez_down <- convert_to_entrez(seed_down)
entrez_bg   <- convert_to_entrez(bg_genes)

ora_up <- ReactomePA::enrichPathway(gene = entrez_up, universe = entrez_bg,
                                    organism = "human", pvalueCutoff = 0.05, readable = TRUE)
ora_down <- ReactomePA::enrichPathway(gene = entrez_down, universe = entrez_bg,
                                      organism = "human", pvalueCutoff = 0.05, readable = TRUE)

cat(sprintf("ORA up-regulated: %d significant pathways\n", nrow(as.data.frame(ora_up))))
cat(sprintf("ORA down-regulated: %d significant pathways\n", nrow(as.data.frame(ora_down))))

In [ ]:
# ============================================================
# 3.6  ORA dotplots (publication-ready, 300 DPI JPG)
# ============================================================
if (nrow(as.data.frame(ora_up)) > 0) {
  p1 <- dotplot(ora_up, showCategory = 15, title = "Reactome -- Seed genes UP in IRI") +
    theme(axis.text.y = element_text(size = 8))
  print(p1)
  ggsave(file.path(OUT_DIR, "dotplot_ORA_up.jpg"), p1,
         width = 10, height = 7, dpi = 300, bg = "white")
}
if (nrow(as.data.frame(ora_down)) > 0) {
  p2 <- dotplot(ora_down, showCategory = 15, title = "Reactome -- Seed genes DOWN in IRI") +
    theme(axis.text.y = element_text(size = 8))
  print(p2)
  ggsave(file.path(OUT_DIR, "dotplot_ORA_down.jpg"), p2,
         width = 10, height = 7, dpi = 300, bg = "white")
}

## 4. Cat. B validation (GSE54888 -- DGF vs IGF)

In [ ]:
# ============================================================
# 4.1  Pre-ranked GSEA of seed genes on Cat. B ranking
#      Tests both the full seed set and the HC subset
# ============================================================
de_b <- de_cat_b[["GSE54888"]] %>%
  dplyr::filter(!is.na(pvalue), pvalue > 0) %>%
  dplyr::mutate(rank_score = sign(log2FoldChange) * (-log10(pvalue))) %>%
  dplyr::distinct(gene, .keep_all = TRUE) %>%
  dplyr::arrange(desc(rank_score))

ranking_b <- setNames(de_b$rank_score, de_b$gene)

seed_sets <- list(
  IRI_seed_genes    = seed_genes$gene,
  IRI_seed_genes_HC = seed_genes_hc$gene
)

gsea_validation <- fgsea::fgsea(pathways = seed_sets, stats = ranking_b,
                                minSize = 5, nPermSimple = 100000)

cat("== GSEA validation on Cat. B (DGF vs IGF) ==\n")
print(gsea_validation[, c("pathway", "pval", "padj", "ES", "NES", "size")])

In [ ]:
# ============================================================
# 4.2  Save validation results
# ============================================================
val_full <- gsea_validation[pathway == "IRI_seed_genes"]
val_hc   <- gsea_validation[pathway == "IRI_seed_genes_HC"]

validation_summary <- data.frame(
  test  = c("Full_GSEA_NES", "Full_GSEA_pval", "Full_N",
            "HC_GSEA_NES", "HC_GSEA_pval", "HC_N"),
  value = c(val_full$NES, val_full$pval, val_full$size,
            val_hc$NES, val_hc$pval, val_hc$size)
)
write.csv(validation_summary, file = file.path(OUT_DIR, "validation_catB_summary.csv"),
          row.names = FALSE)
cat("Validation results saved.\n")

## 5. Co-expression networks (MUUMI Module 2)

We infer separate networks for IRI and Control conditions on two platforms:  
- **GSE43974** (microarray, n=169 IRI + 37 Ctrl) -- robust network  
- **GSE126805** (RNA-seq, n=42 IRI + 41 Ctrl) -- robust network  

GSE142077 (n=5 per condition) is excluded from network inference due to insufficient sample size, but contributes to the statistical meta-analysis.

In [ ]:
# ============================================================
# 5.1  Load expression matrices and sample labels
# ============================================================
load_expr <- function(path) {
  if (!file.exists(path)) { message(sprintf("Not found: %s", path)); return(NULL) }
  as.data.frame(read.csv(path, row.names = 1, check.names = FALSE))
}
load_labels <- function(path) {
  if (!file.exists(path)) return(NULL)
  read.csv(path, stringsAsFactors = FALSE)
}

expr_list   <- lapply(expr_files, load_expr)
labels_list <- lapply(label_files, load_labels)

# Datasets used for network inference (exclude GSE142077: too few samples)
net_datasets <- c("GSE43974", "GSE126805")
for (id in net_datasets) {
  if (is.null(expr_list[[id]])) cat(sprintf("WARNING: %s not available!\n", id))
  else cat(sprintf("%s: %d genes x %d samples\n", id, nrow(expr_list[[id]]), ncol(expr_list[[id]])))
}

In [ ]:
# ============================================================
# 5.2  Subset to seed genes, split by IRI/Control
# ============================================================
prepare_expr_by_condition <- function(expr_df, labels_df, gene_subset) {
  common_genes <- intersect(rownames(expr_df), gene_subset)
  expr_sub <- expr_df[common_genes, , drop = FALSE]
  iri_samples  <- labels_df$sample[labels_df$condition == "IRI"]
  ctrl_samples <- labels_df$sample[labels_df$condition == "Control"]
  list(
    iri     = expr_sub[, intersect(colnames(expr_sub), iri_samples), drop = FALSE],
    control = expr_sub[, intersect(colnames(expr_sub), ctrl_samples), drop = FALSE],
    genes   = common_genes
  )
}

net_gene_set <- seed_genes$gene

prepared <- list()
for (id in net_datasets) {
  if (!is.null(expr_list[[id]]) && !is.null(labels_list[[id]])) {
    prepared[[id]] <- prepare_expr_by_condition(expr_list[[id]], labels_list[[id]], net_gene_set)
    cat(sprintf("%s: %d genes, %d IRI, %d Ctrl\n", id,
                length(prepared[[id]]$genes),
                ncol(prepared[[id]]$iri), ncol(prepared[[id]]$control)))
  }
}

In [ ]:
# ============================================================
# 5.3  Network inference helper function
#      CLR + Pearson correlation -> edge ranking -> binary network
# ============================================================
infer_network <- function(expr_df, label, ncores = 2) {
  cat(sprintf("  Inferring network %s (%d genes x %d samples)...\n",
              label, nrow(expr_df), ncol(expr_df)))
  
  rank_mat <- get_ranked_consensus_matrix(
    gx_table = expr_df, iMethods = c("clr"),
    iEst = c("pearson"), iDisc = c("none"), ncores = ncores
  )
  parsed <- parse_edge_rank_matrix(
    edge_rank_matrix = rank_mat, edge_selection_strategy = "default",
    mat_weights = "rank"
  )
  graph <- get_iGraph(parsed$bin_mat)
  
  cat(sprintf("  -> %d nodes, %d edges\n", vcount(graph), ecount(graph)))
  list(graph = graph, edge_rank = parsed$edge_rank)
}

In [ ]:
# ============================================================
# 5.4  Network inference -- GSE43974 (microarray)
# ============================================================
cat("=== GSE43974 (microarray) ===\n")
net_43974_iri  <- infer_network(prepared[["GSE43974"]]$iri,  "GSE43974_IRI")
net_43974_ctrl <- infer_network(prepared[["GSE43974"]]$control, "GSE43974_Ctrl")

saveRDS(net_43974_iri$graph,  file.path(OUT_DIR, "network_iri_GSE43974.rds"))
saveRDS(net_43974_ctrl$graph, file.path(OUT_DIR, "network_ctrl_GSE43974.rds"))

In [ ]:
# ============================================================
# 5.5  Network inference -- GSE126805 (RNA-seq)
# ============================================================
cat("=== GSE126805 (RNA-seq) ===\n")
net_126805_iri  <- infer_network(prepared[["GSE126805"]]$iri,  "GSE126805_IRI")
net_126805_ctrl <- infer_network(prepared[["GSE126805"]]$control, "GSE126805_Ctrl")

saveRDS(net_126805_iri$graph,  file.path(OUT_DIR, "network_iri_GSE126805.rds"))
saveRDS(net_126805_ctrl$graph, file.path(OUT_DIR, "network_ctrl_GSE126805.rds"))

In [ ]:
# ============================================================
# 5.6  Community detection (walktrap, all 4 networks)
# ============================================================
modules_43974_iri   <- get_modules(net_43974_iri$graph,   method = "walktrap")
modules_43974_ctrl  <- get_modules(net_43974_ctrl$graph,  method = "walktrap")
modules_126805_iri  <- get_modules(net_126805_iri$graph,  method = "walktrap")
modules_126805_ctrl <- get_modules(net_126805_ctrl$graph, method = "walktrap")

cat(sprintf("GSE43974  IRI:  %d modules\n", length(unique(membership(modules_43974_iri)))))
cat(sprintf("GSE43974  Ctrl: %d modules\n", length(unique(membership(modules_43974_ctrl)))))
cat(sprintf("GSE126805 IRI:  %d modules\n", length(unique(membership(modules_126805_iri)))))
cat(sprintf("GSE126805 Ctrl: %d modules\n", length(unique(membership(modules_126805_ctrl)))))

In [ ]:
# ============================================================
# 5.7  Functional annotation (Reactome) -- microarray networks
# ============================================================
cat("=== Pathway enrichment GSE43974 IRI ===\n")
get_reactome_from_modules(modules_43974_iri, geneID = "SYMBOL", pval_cutoff = 0.05,
                          outPath = file.path(OUT_DIR, "pathway_micro_iri"), layout = "overall")

cat("\n=== Pathway enrichment GSE43974 Ctrl ===\n")
get_reactome_from_modules(modules_43974_ctrl, geneID = "SYMBOL", pval_cutoff = 0.05,
                          outPath = file.path(OUT_DIR, "pathway_micro_ctrl"), layout = "overall")

In [ ]:
# ============================================================
# 5.8  Functional annotation -- RNA-seq networks
# ============================================================
cat("=== Pathway enrichment GSE126805 IRI ===\n")
get_reactome_from_modules(modules_126805_iri, geneID = "SYMBOL", pval_cutoff = 0.05,
                          outPath = file.path(OUT_DIR, "pathway_rnaseq_iri"), layout = "overall")

cat("\n=== Pathway enrichment GSE126805 Ctrl ===\n")
get_reactome_from_modules(modules_126805_ctrl, geneID = "SYMBOL", pval_cutoff = 0.05,
                          outPath = file.path(OUT_DIR, "pathway_rnaseq_ctrl"), layout = "overall")

In [ ]:
# ============================================================
# 5.9  Bubble plots (with label wrapping to avoid overlap)
# ============================================================

# Helper: wrap long pathway names to prevent overlapping y-axis labels
wrap_labels <- function(plot_obj, width = 40) {
  plot_obj + scale_y_discrete(labels = function(x) stringr::str_wrap(x, width = width)) +
    theme(axis.text.y = element_text(size = 7, lineheight = 0.9))
}

cat("Bubble plot GSE43974 IRI:\n")
bp_43974_iri <- get_bubbleplot_from_pathways(modules_43974_iri, geneID = "SYMBOL")
bp_43974_iri_fixed <- wrap_labels(bp_43974_iri)
print(bp_43974_iri_fixed)
ggsave(file.path(OUT_DIR, "bubbleplot_micro_iri.jpg"), bp_43974_iri_fixed,
       width = 12, height = 9, dpi = 300, bg = "white")

cat("\nBubble plot GSE126805 IRI:\n")
bp_126805_iri <- get_bubbleplot_from_pathways(modules_126805_iri, geneID = "SYMBOL")
bp_126805_iri_fixed <- wrap_labels(bp_126805_iri)
print(bp_126805_iri_fixed)
ggsave(file.path(OUT_DIR, "bubbleplot_rnaseq_iri.jpg"), bp_126805_iri_fixed,
       width = 12, height = 9, dpi = 300, bg = "white")

## 6. Cross-platform late integration (MUUMI Module 3)

We aggregate co-expression modules from microarray (GSE43974) and RNA-seq (GSE126805) using `aggregate_communities()` from MUUMI, which employs Non-negative Matrix Factorization (NMF) to identify shared meta-modules across platforms (Inkala et al., 2026, Module 3).

In [ ]:
# ============================================================
# 6.1  Late integration -- IRI networks
# ============================================================

# Prepare community labels as named lists (required format)
micro_cl_iri  <- setNames(as.list(membership(modules_43974_iri)),  modules_43974_iri$names)
rnaseq_cl_iri <- setNames(as.list(membership(modules_126805_iri)), modules_126805_iri$names)

community_labels_iri <- list("micro" = micro_cl_iri, "rnaseq" = rnaseq_cl_iri)

# Number of aggregated communities = rounded average
n_micro_iri  <- max(membership(modules_43974_iri))
n_rnaseq_iri <- max(membership(modules_126805_iri))
n_agg_iri    <- round((n_micro_iri + n_rnaseq_iri) / 2)

cat(sprintf("Micro IRI: %d, RNA-seq IRI: %d -> aggregated: %d\n",
            n_micro_iri, n_rnaseq_iri, n_agg_iri))

agg_iri <- aggregate_communities(
  community_labels_iri,
  n_aggregate_communities = n_agg_iri,
  max_iter = 50000
)

most_probable_iri <- get_most_probable_community(agg_iri)
cat(sprintf("Integrated IRI genes: %d in %d aggregated communities\n",
            length(most_probable_iri), length(unique(most_probable_iri))))

In [ ]:
# ============================================================
# 6.2  Late integration -- Control networks
# ============================================================
micro_cl_ctrl  <- setNames(as.list(membership(modules_43974_ctrl)),  modules_43974_ctrl$names)
rnaseq_cl_ctrl <- setNames(as.list(membership(modules_126805_ctrl)), modules_126805_ctrl$names)

community_labels_ctrl <- list("micro" = micro_cl_ctrl, "rnaseq" = rnaseq_cl_ctrl)

n_micro_ctrl  <- max(membership(modules_43974_ctrl))
n_rnaseq_ctrl <- max(membership(modules_126805_ctrl))
n_agg_ctrl    <- round((n_micro_ctrl + n_rnaseq_ctrl) / 2)

cat(sprintf("Micro Ctrl: %d, RNA-seq Ctrl: %d -> aggregated: %d\n",
            n_micro_ctrl, n_rnaseq_ctrl, n_agg_ctrl))

agg_ctrl <- aggregate_communities(
  community_labels_ctrl,
  n_aggregate_communities = n_agg_ctrl,
  max_iter = 50000
)

most_probable_ctrl <- get_most_probable_community(agg_ctrl)
cat(sprintf("Integrated Ctrl genes: %d in %d aggregated communities\n",
            length(most_probable_ctrl), length(unique(most_probable_ctrl))))

In [ ]:
# ============================================================
# 6.3  Platform contributions to each aggregated community
# ============================================================
cat("=== Platform contributions -- IRI communities ===\n")
print(round(agg_iri$view_contributions, 3))

cat("\n=== Platform contributions -- Ctrl communities ===\n")
print(round(agg_ctrl$view_contributions, 3))

In [ ]:
# ============================================================
# 6.4  Functional annotation of integrated modules
# ============================================================
cat("=== Pathway enrichment -- integrated IRI communities ===\n")
get_reactome_from_modules(most_probable_iri, geneID = "SYMBOL", pval_cutoff = 0.05,
                          outPath = file.path(OUT_DIR, "pathway_integrated_iri"), layout = "overall")

cat("\n=== Pathway enrichment -- integrated Ctrl communities ===\n")
get_reactome_from_modules(most_probable_ctrl, geneID = "SYMBOL", pval_cutoff = 0.05,
                          outPath = file.path(OUT_DIR, "pathway_integrated_ctrl"), layout = "overall")

In [ ]:
# ============================================================
# 6.5  Bubble plots of integrated modules (label-wrap fix)
# ============================================================
cat("Bubble plot -- integrated IRI communities:\n")
bp_int_iri <- get_bubbleplot_from_pathways(most_probable_iri, geneID = "SYMBOL")
bp_int_iri_fixed <- wrap_labels(bp_int_iri)
print(bp_int_iri_fixed)
ggsave(file.path(OUT_DIR, "bubbleplot_integrated_iri.jpg"), bp_int_iri_fixed,
       width = 12, height = 9, dpi = 300, bg = "white")

cat("\nBubble plot -- integrated Ctrl communities:\n")
bp_int_ctrl <- get_bubbleplot_from_pathways(most_probable_ctrl, geneID = "SYMBOL")
bp_int_ctrl_fixed <- wrap_labels(bp_int_ctrl)
print(bp_int_ctrl_fixed)
ggsave(file.path(OUT_DIR, "bubbleplot_integrated_ctrl.jpg"), bp_int_ctrl_fixed,
       width = 12, height = 9, dpi = 300, bg = "white")

## 7. ESEA -- Edge Set Enrichment Analysis (MUUMI)

ESEA evaluates functional enrichment at the **edge** level (gene-gene interactions), not individual genes. This is a unique contribution of MUUMI: it identifies pathways whose genes are highly interconnected in the network, revealing functional coordination (Inkala et al., 2026, Module 4).

In [ ]:
# ============================================================
# 7.1  Build pathway edge map from KEGG
# ============================================================
if (!requireNamespace("msigdbr", quietly = TRUE)) install.packages("msigdbr")
library(msigdbr)

# msigdbr v10+: parameter names may differ
kegg <- tryCatch(
  msigdbr(species = "Homo sapiens", category = "C2", subcategory = "CP:KEGG"),
  error = function(e) {
    message("Trying updated parameter names...")
    msigdbr(species = "Homo sapiens", collection = "C2", subcollection = "CP:KEGG_MEDICUS")
  }
)
kegg_pathways <- split(kegg$gene_symbol, kegg$gs_name)
cat(sprintf("KEGG pathways loaded: %d\n", length(kegg_pathways)))

# Generate all gene-pair edges for each pathway
build_pathway_edge_map <- function(pw_list, network_genes, min_size = 3, max_size = 200) {
  edge_map <- list()
  for (pw_name in names(pw_list)) {
    genes <- intersect(pw_list[[pw_name]], network_genes)
    if (length(genes) >= min_size && length(genes) <= max_size) {
      pairs <- combn(sort(genes), 2)
      edges <- paste(pairs[1, ], pairs[2, ], sep = "|")
      edge_map[[pw_name]] <- edges
    }
  }
  edge_map
}

net_genes_iri <- V(net_43974_iri$graph)$name
pathway_edge_map <- build_pathway_edge_map(kegg_pathways, net_genes_iri)
cat(sprintf("Pathway edge map: %d pathways with valid edge sets\n", length(pathway_edge_map)))

In [ ]:
# ============================================================
# 7.2  Prepare edge score vector from IRI network
#      Score = inverse rank (top-ranked edges get highest score)
# ============================================================
edge_rank_iri <- net_43974_iri$edge_rank
edge_rank_iri <- gsub(";", "|", edge_rank_iri, fixed = TRUE)

esea_scores_iri <- rev(seq_along(edge_rank_iri))
names(esea_scores_iri) <- edge_rank_iri

cat(sprintf("Edge score vector: %d edges\n", length(esea_scores_iri)))
cat(sprintf("Top 5 edges: %s\n", paste(head(names(esea_scores_iri)), collapse = ", ")))

In [ ]:
# ============================================================
# 7.3  ESEA -- IRI network (GSE43974)
# ============================================================
esea_iri <- compute_esea(
  EdgeCorScore     = esea_scores_iri,
  pathway_edge_map = pathway_edge_map,
  weighted.score.type = 1,
  nperm            = 1000,
  FDR.threshold    = 0.05,
  topgs            = 5
)

cat("=== ESEA results -- IRI network (top of rank) ===\n")
esea_iri_pos <- esea_iri$SUMMARY.RESULTS$SUMMARY.RESULTS.High_portion_of_the_rank
if (nrow(esea_iri_pos) > 0) {
  print(head(esea_iri_pos[, c("GS", "SIZE", "ES", "NES", "NOM_pval", "FDR_qval")], 10))
} else {
  cat("No significantly enriched edge sets.\n")
}

In [ ]:
# ============================================================
# 7.4  ESEA -- Control network (GSE43974)
# ============================================================
edge_rank_ctrl <- net_43974_ctrl$edge_rank
edge_rank_ctrl <- gsub(";", "|", edge_rank_ctrl, fixed = TRUE)
esea_scores_ctrl <- rev(seq_along(edge_rank_ctrl))
names(esea_scores_ctrl) <- edge_rank_ctrl

# Rebuild edge map for Control network genes
net_genes_ctrl <- V(net_43974_ctrl$graph)$name
pathway_edge_map_ctrl <- build_pathway_edge_map(kegg_pathways, net_genes_ctrl)

esea_ctrl <- compute_esea(
  EdgeCorScore     = esea_scores_ctrl,
  pathway_edge_map = pathway_edge_map_ctrl,
  weighted.score.type = 1,
  nperm            = 1000,
  FDR.threshold    = 0.05,
  topgs            = 5
)

cat("=== ESEA results -- Control network (top of rank) ===\n")
esea_ctrl_pos <- esea_ctrl$SUMMARY.RESULTS$SUMMARY.RESULTS.High_portion_of_the_rank
if (nrow(esea_ctrl_pos) > 0) {
  print(head(esea_ctrl_pos[, c("GS", "SIZE", "ES", "NES", "NOM_pval", "FDR_qval")], 10))
} else {
  cat("No significantly enriched edge sets.\n")
}

In [ ]:
# ============================================================
# 7.5  Save ESEA results
# ============================================================
if (nrow(esea_iri_pos) > 0) {
  write.csv(esea_iri_pos, file = file.path(OUT_DIR, "esea_iri_results.csv"), row.names = FALSE)
}
if (nrow(esea_ctrl_pos) > 0) {
  write.csv(esea_ctrl_pos, file = file.path(OUT_DIR, "esea_ctrl_results.csv"), row.names = FALSE)
}
cat("ESEA results saved.\n")

## 8. Summary and publication-ready figures

In [ ]:
# ============================================================
# 8.1  Hub genes and convergence with seed genes
# ============================================================
hub_genes_iri <- get_ranked_gene_list(
  iGraph = net_43974_iri$graph,
  rank_list_attr = c("betweenness", "degree", "closeness", "eigenvector")
)

cat("Top 20 hub genes in IRI network (GSE43974):\n")
print(head(hub_genes_iri, 20))

# Convergence: hub genes also in high-confidence seed set
hub_top50 <- head(hub_genes_iri, 50)
convergence_hc <- intersect(hub_top50, seed_genes_hc$gene)
cat(sprintf("\nConvergence hub top-50 & seed HC: %d genes\n", length(convergence_hc)))
if (length(convergence_hc) > 0) cat(paste(convergence_hc, collapse = ", "), "\n")

In [ ]:
# ============================================================
# 8.2  Final summary
# ============================================================
cat("============================================================\n")
cat("       MAGIC SOLUTION -- Phase 2: Summary\n")
cat("============================================================\n\n")

cat(sprintf("Cat. A datasets: %d (meta-analysis) | Cat. B: %d (validation)\n",
            length(de_cat_a), length(de_cat_b)))
cat(sprintf("Genes in meta-analysis: %d\n", nrow(meta_rank)))
cat(sprintf("GSEA-based threshold: top %d genes\n", threshold))
cat(sprintf("\nSeed genes (|lfc|>0.379, consist.>0.5): %d (UP: %d, DOWN: %d)\n",
            nrow(seed_genes), sum(seed_genes$weighted_lfc > 0), sum(seed_genes$weighted_lfc < 0)))
cat(sprintf("Seed genes HIGH-CONFIDENCE (consist.=1.0): %d (UP: %d, DOWN: %d)\n",
            nrow(seed_genes_hc), sum(seed_genes_hc$weighted_lfc > 0), sum(seed_genes_hc$weighted_lfc < 0)))

cat(sprintf("\nCat. B validation (GSE54888):\n"))
cat(sprintf("  Full set:  NES=%.3f, p=%.2e (n=%d)\n", val_full$NES, val_full$pval, val_full$size))
cat(sprintf("  HC set:    NES=%.3f, p=%.2e (n=%d)\n", val_hc$NES, val_hc$pval, val_hc$size))

cat(sprintf("\nCo-expression networks:\n"))
cat(sprintf("  GSE43974 (micro)    IRI: %d nodes, %d edges, %d modules\n",
            vcount(net_43974_iri$graph), ecount(net_43974_iri$graph),
            length(unique(membership(modules_43974_iri)))))
cat(sprintf("  GSE43974 (micro)    Ctrl: %d nodes, %d edges, %d modules\n",
            vcount(net_43974_ctrl$graph), ecount(net_43974_ctrl$graph),
            length(unique(membership(modules_43974_ctrl)))))
cat(sprintf("  GSE126805 (RNA-seq) IRI: %d nodes, %d edges, %d modules\n",
            vcount(net_126805_iri$graph), ecount(net_126805_iri$graph),
            length(unique(membership(modules_126805_iri)))))
cat(sprintf("  GSE126805 (RNA-seq) Ctrl: %d nodes, %d edges, %d modules\n",
            vcount(net_126805_ctrl$graph), ecount(net_126805_ctrl$graph),
            length(unique(membership(modules_126805_ctrl)))))

cat(sprintf("\nLate integration (NMF):\n"))
cat(sprintf("  IRI:  %d aggregated communities\n", n_agg_iri))
cat(sprintf("  Ctrl: %d aggregated communities\n", n_agg_ctrl))

cat(sprintf("\nESEA: %d IRI pathways, %d Ctrl pathways (FDR<0.05)\n",
            sum(esea_iri_pos$FDR_qval < 0.05),
            sum(esea_ctrl_pos$FDR_qval < 0.05)))

cat("\n============================================================\n")
cat("Output saved in:", OUT_DIR, "\n")
cat("============================================================\n")

In [ ]:
# ============================================================
# 8.3  Volcano plot of meta-analysis results
# ============================================================
volcano_df <- meta_rank %>%
  dplyr::filter(!is.na(weighted_lfc), !is.na(direction_consistency)) %>%
  dplyr::mutate(
    neg_log10_rank = -log10(rank / max(rank)),
    category = case_when(
      gene %in% seed_genes_hc$gene & weighted_lfc > 0 ~ "Up (high-confidence)",
      gene %in% seed_genes_hc$gene & weighted_lfc < 0 ~ "Down (high-confidence)",
      gene %in% seed_genes$gene & weighted_lfc > 0    ~ "Up",
      gene %in% seed_genes$gene & weighted_lfc < 0    ~ "Down",
      TRUE ~ "Not significant"
    )
  )

top_labels <- volcano_df %>%
  dplyr::filter(category != "Not significant") %>%
  dplyr::arrange(rank) %>%
  dplyr::slice_head(n = 20)

p_volcano <- ggplot(volcano_df, aes(x = weighted_lfc, y = neg_log10_rank, color = category)) +
  geom_point(alpha = 0.4, size = 0.8) +
  geom_point(data = volcano_df %>% filter(category != "Not significant"),
             alpha = 0.7, size = 1.2) +
  scale_color_manual(
    values = c("Not significant" = "grey75",
               "Up" = "#E64B35",
               "Down" = "#4DBBD5",
               "Up (high-confidence)" = "#B2182B",
               "Down (high-confidence)" = "#2166AC"),
    name = "Category"
  ) +
  geom_vline(xintercept = c(-0.379, 0.379), linetype = "dashed", color = "grey40", linewidth = 0.3) +
  ggrepel::geom_text_repel(
    data = top_labels, aes(label = gene),
    size = 2.5, max.overlaps = 20, segment.size = 0.2, color = "black"
  ) +
  labs(
    x = expression("Weighted log"[2]*" fold change"),
    y = expression("-log"[10]*"(normalized rank)"),
    title = "Meta-analysis volcano plot",
    subtitle = sprintf("3 datasets | %d seed genes (%d high-confidence)", nrow(seed_genes), nrow(seed_genes_hc))
  ) +
  theme_bw(base_size = 11) +
  theme(
    legend.position = c(0.85, 0.82),
    legend.background = element_rect(fill = "white", color = "grey80"),
    legend.text = element_text(size = 8),
    legend.title = element_text(size = 9, face = "bold"),
    plot.title = element_text(size = 13, face = "bold"),
    plot.subtitle = element_text(size = 9, color = "grey40")
  )

print(p_volcano)
ggsave(file.path(OUT_DIR, "volcano_meta_analysis.jpg"), p_volcano,
       width = 7, height = 5.5, dpi = 300, bg = "white")

In [ ]:
# ============================================================
# 8.4  Platform contribution barplot (late integration)
# ============================================================
build_contrib_df <- function(agg_result, condition_label) {
  contrib <- agg_result$view_contributions
  df <- as.data.frame(contrib)
  df$platform <- rownames(df)
  df_long <- tidyr::pivot_longer(df, cols = -platform, names_to = "community", values_to = "contribution")
  df_long$community <- gsub("V", "Community ", df_long$community)
  df_long$condition <- condition_label
  df_long$platform <- ifelse(df_long$platform == "micro", "Microarray (GSE43974)", "RNA-seq (GSE126805)")
  df_long
}

contrib_df <- rbind(
  build_contrib_df(agg_iri, "IRI"),
  build_contrib_df(agg_ctrl, "Control")
)

p_contrib <- ggplot(contrib_df, aes(x = contribution, y = community, fill = platform)) +
  geom_bar(stat = "identity", position = "stack", width = 0.7) +
  facet_wrap(~ condition, scales = "free_y") +
  scale_fill_manual(
    values = c("Microarray (GSE43974)" = "#E07A5F", "RNA-seq (GSE126805)" = "#3D85C6"),
    name = "Data source"
  ) +
  labs(
    x = "Contribution strength",
    y = "Integrated community",
    title = "Platform contributions to integrated communities",
    subtitle = "NMF-based late integration of microarray and RNA-seq co-expression networks"
  ) +
  theme_bw(base_size = 11) +
  theme(
    legend.position = "bottom",
    strip.text = element_text(size = 11, face = "bold"),
    plot.title = element_text(size = 13, face = "bold"),
    plot.subtitle = element_text(size = 9, color = "grey40"),
    panel.grid.minor = element_blank()
  )

print(p_contrib)
ggsave(file.path(OUT_DIR, "platform_contributions.jpg"), p_contrib,
       width = 9, height = 5, dpi = 300, bg = "white")

In [ ]:
# ============================================================
# 8.5  Heatmap of log2FC consistency across datasets
# ============================================================
top_n <- 50
top_seeds <- seed_genes %>% dplyr::slice_head(n = top_n)

lfc_for_heatmap <- lapply(names(de_cat_a), function(id) {
  d <- de_cat_a[[id]] %>%
    dplyr::select(gene, log2FoldChange) %>%
    dplyr::distinct(gene, .keep_all = TRUE)
  colnames(d)[2] <- id
  d
})

hm_df <- lfc_for_heatmap[[1]]
for (i in 2:length(lfc_for_heatmap)) {
  hm_df <- merge(hm_df, lfc_for_heatmap[[i]], by = "gene", all = FALSE)
}

hm_df <- hm_df %>% dplyr::filter(gene %in% top_seeds$gene)
rownames(hm_df) <- hm_df$gene
hm_mat <- as.matrix(hm_df[, -1])

gene_order <- top_seeds$gene[top_seeds$gene %in% rownames(hm_mat)]
hm_mat <- hm_mat[gene_order, ]

hm_long <- reshape2::melt(hm_mat)
colnames(hm_long) <- c("Gene", "Dataset", "log2FC")
hm_long$log2FC_trunc <- pmax(pmin(hm_long$log2FC, 3), -3)
hm_long$HC <- ifelse(hm_long$Gene %in% seed_genes_hc$gene, "*", "")

p_heatmap <- ggplot(hm_long, aes(x = Dataset, y = Gene, fill = log2FC_trunc)) +
  geom_tile(color = "white", linewidth = 0.3) +
  geom_text(aes(label = HC), size = 3, color = "black", vjust = 0.8) +
  scale_fill_gradient2(
    low = "#2166AC", mid = "white", high = "#B2182B", midpoint = 0,
    limits = c(-3, 3), name = expression("log"[2]*"FC"),
    oob = scales::squish
  ) +
  scale_y_discrete(limits = rev(gene_order)) +
  labs(
    x = NULL,
    y = NULL,
    title = sprintf("Top %d seed genes -- log2FC across datasets", top_n),
    subtitle = "* = high-confidence (consistent direction in all 3 datasets)"
  ) +
  theme_minimal(base_size = 10) +
  theme(
    axis.text.y = element_text(size = 6, face = "plain"),
    axis.text.x = element_text(size = 10, angle = 30, hjust = 1, face = "bold"),
    plot.title = element_text(size = 13, face = "bold"),
    plot.subtitle = element_text(size = 9, color = "grey40"),
    legend.position = "right",
    panel.grid = element_blank()
  )

print(p_heatmap)
ggsave(file.path(OUT_DIR, "heatmap_top_seed_genes.jpg"), p_heatmap,
       width = 5, height = 10, dpi = 300, bg = "white")

In [ ]:
sessionInfo()